# In silico perturbation experiment

In [2]:
from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats

from geneformer.tokenizer import TOKEN_DICTIONARY_FILE
from geneformer.in_silico_perturber_stats import GENE_NAME_ID_DICTIONARY_FILE
from geneformer.in_silico_perturber import ISP_device

import sys

print(f"usse gpu num: {ISP_device}")
print(f"gene to ens dict: {GENE_NAME_ID_DICTIONARY_FILE}")
print(f"token directory : {TOKEN_DICTIONARY_FILE}")

import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0)) 


usse gpu num: cuda:0
gene to ens dict: /app/geneformer/dicts/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl
token directory : /app/geneformer/dicts/MLM-re_token_dictionary_v1.pkl
True
NVIDIA GB10


In [13]:
from datasets import load_from_disk
import numpy as np


# load disease dataset (xxx.dataset)
dataset_name = "/app/data/Mouse-Genecorpus-20M/eval_dataset/in_silico_perturbation/Cop1KO_isp_mouse_tokenize_dataset_v-n1.dataset"


data = load_from_disk(dataset_name)
print(data)
print("total cells: {}".format(len(data["length"])))

disease_types = np.unique(data["disease"])
print(disease_types)
print(disease_types.shape[0])




Dataset({
    features: ['input_ids', 'cell_types', 'organ_major', 'disease', 'length'],
    num_rows: 12441
})
total cells: 12441
['Cop1_KO' 'Cop1_WT']
2


In [14]:
# in silico perturbation in deletion mode to determine genes whose 
# deletion in the dilated cardiomyopathy (dcm) state significantly shifts
# the embedding towards non-failing (nf) state


# select_perturb_type: "delete","overexpress","inhibit","activate"

# "delete": delete gene from rank value encoding
# "overexpress": move gene to front of rank value encoding
# "inhibit": move gene to lower quartile of rank value encoding
# "activate": move gene to higher quartile of rank value encoding


select_perturb_type = "delete"

start_state = "Cop1_WT"          # <-- change this
end_state = "Cop1_KO"            # <-- change this
alt_state = []

use_model_type = "Pretrained"    # <-- change this (you have the pretrained model, not a fine-tuned CellClassifier)

genes_to_perturb_list = []

isp = InSilicoPerturber(perturb_type=select_perturb_type,
                        perturb_rank_shift=None,
                        genes_to_perturb="all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list,
                        combos=0,
                        anchor_gene=None,
                        model_type=use_model_type,
                        num_classes=2,               # <-- 2 classes: Cop1_WT and Cop1_KO
                        emb_mode="cell",
                        cell_emb_style="mean_pool",
                        filter_data=None,
                        cell_states_to_model={'state_key': 'disease', 
                                              'start_state': start_state, 
                                              'goal_state': end_state, 
                                              'alt_states': alt_state}, 
                        max_ncells=2000,
                        emb_layer=0,
                        forward_batch_size=50,
                        nproc=6)




In [16]:
# outputs intermediate files from in silico perturbation
start_state = start_state.replace(" ","-")
end_state = end_state.replace(" ","-")

import os   
DIR_NAME = "/app/output/isp_results"
os.makedirs(DIR_NAME, exist_ok=True)

# ADD THIS LINE:
organ_data = "Cop1KO"

isp.perturb_data("/app/models/mouse-Geneformer/",
                 dataset_name,
                 DIR_NAME + "/",
                 "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(select_perturb_type, organ_data, start_state, end_state))





Filter (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/12441 [00:00<?, ? examples/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

/app/geneformer/in_silico_perturber.py:97: SyntaxWarning: invalid escape sequence '\('
  return int(re.split("\(|,",str(model.bert.embeddings.position_embeddings))[1])


AttributeError: 'Column' object has no attribute 'to'

In [ ]:


ispstats = InSilicoPerturberStats(mode="goal_state_shift",
                                  genes_perturbed="all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list,
                                  combos=0,
                                  anchor_gene=None,
                                  cell_states_to_model={"state_key": "disease",
                                                        "start_state": start_state,
                                                        "goal_state": end_state,
                                                        "alt_states": alt_state},
                                 )


In [ ]:
# extracts data from intermediate files and processes stats to output in final .csv

start_state = start_state.replace(" ","-")
end_state = end_state.replace(" ","-")

import os
DIR_NAME = "/path/to/save/ispstats.get_stats/result"
if not os.path.exists(DIR_NAME):
    os.mkdir(DIR_NAME)


ispstats.get_stats("/path/to/save/isp.perturb_data/result/", # path to input data
                   None,
                   "/path/to/save/ispstats.get_stats/result/", 
                   "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(select_perturb_type, organ_data, start_state, end_state)) # output prefix 
